# NISAR atlas · Ostrava–Karviná: checks of the thresholds

**Why this notebook.** Some thresholds of the main notebook (`ostrava_karvina.ipynb`) were chosen or changed **after**
looking at the results (see `notes/experiments.md`, block O, O2 and O2b). Each key threshold is checked here by tests that do
not depend on the wanted answer:

| Test | What it shows | Thresholds |
|---|---|---|
| **A. Frame coverage** | the coverage distribution has two humps, so any threshold between them selects the same frames | `MIN_COVERAGE` |
| **B. Coherence vs noise** | how the rate noise grows as coherence falls (+ theoretical shape) | `COH_MIN` |
| **C. Vertical noise and variogram** | noise level in stable places (σ) and the size of atmospheric patches | `VERT_ZONE_CM`, `HP_SIGMA_KM` |
| **D. Null test (sign flips)** | how many bowls the method finds where there is no motion by construction → false discovery rate (FDR) | `VERT_ZONE_CM`, `MIN_CONFIRM_TRACKS`, `LOS_CONFIRM_CM` |
| **E. Injected bowls** | which share of real bowls of a given strength the method finds (recall) | `VERT_ZONE_CM`, `MIN_ZONE_PX` |
| **F. Noise filter** | whether the long-wavelength filter eats bowls, with and without the mask | `HP_SIGMA_KM`, `HP_MASK_CM` |
| **G. Split plateau** | how the bowl count depends on the saddle depth | `SPLIT_DEPTH_CM` |
| **H. Robustness** | whether the same 3-track bowls stay when each parameter is changed on its own | all clustering parameters |

**Input:** `output/ostrava_karvina/stack_*.nc` of the three frames and, if present, `osm_water.geojson` (both written by
`ostrava_karvina.ipynb`). Nothing is downloaded. **Output:** `output/ostrava_karvina/thresholds/` (ignored by git).
**Memory** < 1.5 GB. **Time** ~5–15 min on a CPU (depends on `N_PERM`).

**Run** from the repo root (versions pinned, no change to the project lock):

```
uv run --with jupyterlab==4.6.6 --with xarray==2026.9.0 --with rioxarray==0.23.0 --with netcdf4==1.7.5.1.post1 \
       --with geopandas==1.2.0 --with scikit-image==0.26.0 --with pandas==3.1.1 jupyter lab scripts/ostrava/ostrava_thresholds.ipynb
```

**Caveat.** The stacks are saved after the deramp, the noise filter (σ = 5 km) and the coherence mask 0.3. Tests B, C and F
therefore show the behaviour *after* this processing, and thresholds below 0.3 cannot be checked here (see test B).

**Status.** These checks were run after the thresholds were chosen; they test the choices, they were not pre-registered.
The detector below is a copy of the main notebook's bowl logic; with the water mask it gives a slightly different bowl count
than the first run (see the printout of section 2).

## 0. Settings

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, rioxarray, warnings, json, time
import matplotlib.pyplot as plt
from scipy import ndimage
from rasterio.enums import Resampling
from skimage.segmentation import watershed
from skimage.morphology import h_maxima
warnings.filterwarnings("ignore")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "output" / "ostrava_karvina"          # written by ostrava_karvina.ipynb
OUT = DATA / "thresholds"
OUT.mkdir(parents=True, exist_ok=True)
if not sorted(DATA.glob("stack_*.nc")):
    raise FileNotFoundError(f"No stack_*.nc in {DATA.relative_to(ROOT)}: run scripts/ostrava/ostrava_karvina.ipynb first")
KA, KD = "A158_F027", "D152_F063"          # the pair used for the vertical (as in the main notebook)
USE_WATER = True                           # the analyst's first run of these checks had no water file (water 0 %)
WATER_FILE = DATA / "osm_water.geojson"
N_PERM = 30                                # permutations of the null test (more = more precise, slower)
RNG = np.random.default_rng(42)            # fixed seed for reproducibility

# defaults of the main notebook (the values under test)
P0 = dict(VERT_ZONE_CM=-15, MIN_VALID_FRAC=0.6, CLUSTER_GAP_M=400, MAX_HOLE_KM2=3, MAX_WATER_FRAC=0.30,
          SPLIT_DEPTH_CM=10, SPLIT_SMOOTH_PX=2, MIN_ZONE_PX=20, LOS_CONFIRM_CM=-8, MIN_CONFIRM_TRACKS=2)

stacks = {p.stem.replace("stack_", ""): xr.open_dataset(p).load() for p in sorted(DATA.glob("stack_*.nc"))}
print("Stacks:", {k: dict(v.sizes) for k, v in stacks.items()})
assert KA in stacks and KD in stacks, f"Missing {KA} or {KD}; set KA/KD to one of {list(stacks)}"

## 1. Rates and the vertical (as in the main notebook)

In [ ]:
def rates(ds):
    return (ds.disp / ds.dt_days * 365.25 * 100).values          # cm/yr for each pair

def velocity(R, min_valid=P0["MIN_VALID_FRAC"]):
    valid = np.isfinite(R).mean(0)
    v = np.nanmean(R, 0)
    v[valid < min_valid] = np.nan
    return v.astype("f4")

def velocity_null(R, signs, min_valid=P0["MIN_VALID_FRAC"]):
    # "null" rate: first subtract EACH pixel's mean rate (constant motion vanishes exactly, whatever the number of pairs
    # and gaps), then random signs on the deviations. The factor sqrt(n/(n-1)) gives the noise the same size as in the
    # real mean.
    ok = np.isfinite(R); n = ok.sum(0).astype("f8")
    res = R - np.nanmean(R, 0)
    v = np.nansum(res * signs[:, None, None], 0) / np.maximum(n, 1) * np.sqrt(n / np.maximum(n - 1, 1))
    v[(ok.mean(0) < min_valid) | (n < 2)] = np.nan
    return v.astype("f4")

def da(arr, ds):
    a = xr.DataArray(np.asarray(arr, "f4"), dims=("y", "x"), coords=dict(y=ds.y, x=ds.x))
    return a.rio.write_crs(int(ds.attrs["epsg"])).rio.write_nodata(np.nan)

dA, dD = stacks[KA], stacks[KD]
RA, RD = rates(dA), rates(dD)
others = [k for k in stacks if k not in (KA, KD)]
R_other = {k: rates(stacks[k]) for k in others}
grid = da(np.zeros((dA.sizes["y"], dA.sizes["x"])), dA)
PX = float(abs(dA.x[1] - dA.x[0])); PX_KM2 = (PX / 1000) ** 2
eA, uA = dA.los_x.values, dA.los_z.values
on_grid = lambda arr, ds: da(arr, ds).rio.reproject_match(grid, resampling=Resampling.bilinear).values
eD, uD = on_grid(dD.los_x.values, dD), on_grid(dD.los_z.values, dD)
det = eA * uD - uA * eD

def vertical(vA, vD_grid):
    U = (eA * vD_grid - vA * eD) / det
    U[np.abs(det) < 0.2] = np.nan
    return U.astype("f4")

def los_maps(vA, vD_grid, v_others_grid):
    return {KA: vA, KD: vD_grid, **v_others_grid}

vA = velocity(RA); vDg = on_grid(velocity(RD), dD)
vOg = {k: on_grid(velocity(R_other[k]), stacks[k]) for k in others}
U = vertical(vA, vDg)
L_real = los_maps(vA, vDg, vOg)

# water mask
water = np.zeros(U.shape, bool)
if USE_WATER and WATER_FILE.exists():
    import geopandas as gpd
    from rasterio import features
    g = gpd.read_file(WATER_FILE).to_crs(grid.rio.crs)
    g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    water = features.rasterize(((geom, 1) for geom in g.geometry), out_shape=U.shape,
                               transform=grid.rio.transform(), fill=0, all_touched=True).astype(bool)
print(f"Pixel {PX:.0f} m · vertical: {np.isfinite(U).sum()} pixels · water: {water.mean():.1%}")

## 2. Bowl detector (copy of the main notebook's logic; parameters can be changed)

In [ ]:
def detect(U, L, p=P0, water=water):
    thr = p["VERT_ZONE_CM"]
    Us = ndimage.median_filter(np.nan_to_num(U), size=3)
    mask = ndimage.binary_opening(Us < thr) & ~water
    nodata = ~np.isfinite(U) & ~water
    nl, nn = ndimage.label(nodata)
    if nn:
        idx = np.arange(1, nn + 1)
        near = ndimage.binary_dilation(mask, iterations=2)
        sizes = ndimage.sum(nodata, nl, idx) * PX_KM2
        touch = ndimage.maximum(near, nl, idx)
        mask |= np.isin(nl, idx[(sizes < p["MAX_HOLE_KM2"]) & (touch > 0)])
    r = max(int(round(p["CLUSTER_GAP_M"] / 2 / PX)), 1)
    disk = np.hypot(*np.mgrid[-r:r+1, -r:r+1]) <= r
    merged = ndimage.binary_fill_holes(ndimage.binary_closing(mask, structure=disk))
    base, nb = ndimage.label(merged)
    if nb == 0: return pd.DataFrame()
    Uf = U.copy(); nanm = ~np.isfinite(Uf)
    if nanm.any():
        ii = ndimage.distance_transform_edt(nanm, return_distances=False, return_indices=True); Uf = Uf[tuple(ii)]
    Uf = ndimage.gaussian_filter(Uf, p["SPLIT_SMOOTH_PX"])
    mk = h_maxima(-Uf, p["SPLIT_DEPTH_CM"]).astype(bool) & merged & (Uf < thr)
    markers, nmk = ndimage.label(mk)
    has = set(np.unique(base[markers > 0])) - {0}
    miss = [z for z in range(1, nb + 1) if z not in has]
    if miss:
        pos = ndimage.minimum_position(Uf, base, miss)
        for (iy, ix) in (pos if isinstance(pos, list) else [pos]):
            nmk += 1; markers[iy, ix] = nmk
    lab = watershed(Uf, markers, mask=merged)
    ids = np.unique(lab); ids = ids[ids > 0]
    area = ndimage.sum(np.ones_like(lab), lab, ids)
    wfrac = ndimage.sum(water, lab, ids) / area
    valid = np.isfinite(U) & ~water
    nval = ndimage.sum(valid, lab, ids)
    umean = ndimage.sum(np.where(valid, U, 0), lab, ids) / np.maximum(nval, 1)
    conf = np.zeros(len(ids), int)
    for k, v in L.items():
        ok = np.isfinite(v)
        s = ndimage.sum(np.where(ok, v, 0), lab, ids); c = ndimage.sum(ok, lab, ids)
        conf += ((c >= 5) & (s / np.maximum(c, 1) < p["LOS_CONFIRM_CM"])).astype(int)
    com = ndimage.center_of_mass(np.ones_like(lab), lab, ids)
    df = pd.DataFrame(dict(lab=ids, area_px=area, water_frac=wfrac, n_valid=nval, up_mean=umean, n_conf=conf,
                           cy=[c[0] for c in com], cx=[c[1] for c in com]))
    keep = ((df.area_px >= p["MIN_ZONE_PX"]) & (df.water_frac <= p["MAX_WATER_FRAC"]) &
            (df.n_valid >= p["MIN_ZONE_PX"] // 2) & (df.n_conf >= min(p["MIN_CONFIRM_TRACKS"], len(L))))
    return df[keep].sort_values("up_mean").reset_index(drop=True)

t = time.time(); REF = detect(U, L_real)
print(f"Bowls with the defaults: {len(REF)} (confirmed by 3 tracks: {(REF.n_conf >= 3).sum()}) · {time.time()-t:.1f} s")
REF.head(12)

## A. `MIN_COVERAGE`: frame coverage
Dropped frames were not saved, so their values come from the main notebook's run log (section 4). If the distribution has two
humps, the threshold is not chosen "for the result": any value in the gap between the humps selects the same frames.

In [ ]:
cov_log = {"D051_F063": 0.01, "D051_F064": 0.01, "A158_F026": 0.00, "D080_F063": 0.21,   # medians from the run log
           "A057_F027": None, "D152_F063": None, "A158_F027": None}
for k, ds in stacks.items():                                   # the saved frames: computed from the data
    cov_log[k] = float(np.median(np.isfinite(ds.disp.values).mean(axis=(1, 2))))
cv = pd.Series(cov_log).dropna().sort_values()
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(cv.index, cv.values, color=["#c44" if v < 0.3 else "#4a4" for v in cv.values])
ax.axvline(0.15, ls=":", c="gray", label="first threshold 0.15"); ax.axvline(0.30, ls="--", c="k", label="current threshold 0.30")
lo, hi = cv[cv < 0.3].max(), cv[cv >= 0.3].min()
ax.axvspan(lo, hi, color="gold", alpha=.25, label=f"any threshold here → same frames ({lo:.2f}–{hi:.2f})")
ax.set_xlabel("median share of valid pixels in the box"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(OUT / "A_coverage.png", dpi=130); plt.show()
print(f"Gap between the humps: {lo:.2f} … {hi:.2f}. The threshold 0.15 kept D080 (0.21), which lies below the gap, "
      "so it had to be raised. Any value inside the gap gives the same result.")

## B. `COH_MIN`: coherence vs noise
For each pixel: the spread of the pair rates (std) and the mean coherence. The spread measures noise: real motion is almost
the same over 12 days, noise differs from pair to pair. Theory: σφ = √(1−γ²) / (γ·√(2L)), L = number of looks.
Pixels with γ < 0.3 are already masked in the stack, so the curve exists only for γ ≥ 0.3. To see below, run the main
notebook once with `COH_MIN = 0.1` and re-run this cell.

In [ ]:
rows = []
for k, ds in stacks.items():
    R = rates(ds); g = ds.coh.values.copy(); g[~np.isfinite(R)] = np.nan
    gm = np.nanmean(g, 0); sd = np.nanstd(R, 0); n = np.isfinite(R).sum(0)
    m = (n >= 5) & np.isfinite(gm) & np.isfinite(sd)
    rows.append(pd.DataFrame(dict(track=k, coh=gm[m], std=sd[m])))
df = pd.concat(rows)
bins = np.arange(0.0, 1.01, 0.05); df["bin"] = pd.cut(df.coh, bins)
stat = df.groupby(["track", "bin"], observed=True)["std"].median().unstack(0)
centers = [b.mid for b in stat.index]
fig, ax = plt.subplots(figsize=(8, 4))
for c in stat.columns: ax.plot(centers, stat[c], "o-", label=c)
# theoretical shape (scaled to match at γ = 0.7)
gg = np.linspace(0.2, 0.95, 50); crb = np.sqrt(1 - gg**2) / gg
ref = np.nanmedian(stat.loc[[b for b in stat.index if 0.65 < b.mid < 0.75]].values)
ax.plot(gg, crb / (np.sqrt(1 - .7**2) / .7) * ref, "k--", label="theory √(1−γ²)/γ (shape)")
ax.axvline(0.3, c="r", ls=":", label="COH_MIN = 0.3")
ax.set_xlabel("mean coherence of the pixel"); ax.set_ylabel("spread of pair rates, cm/yr (median)")
ax.legend(fontsize=8); ax.set_ylim(0, None); plt.tight_layout(); plt.savefig(OUT / "B_coherence_noise.png", dpi=130); plt.show()
s03 = np.nanmedian(stat.loc[[b for b in stat.index if 0.3 <= b.left < 0.35]].values)
s07 = np.nanmedian(stat.loc[[b for b in stat.index if 0.7 <= b.left < 0.75]].values)
print(f"Noise at γ≈0.3: {s03:.0f} cm/yr per pair, at γ≈0.7: {s07:.0f} cm/yr ({s03/s07:.1f}× larger). "
      f"Theory between 0.3 and 0.7 gives ×{(np.sqrt(1-.3**2)/.3)/(np.sqrt(1-.7**2)/.7):.1f}.")

## C. Vertical noise (σ) and variogram
**Stable pixels** are those more than 1 km from any detected bowl. Their robust spread (1.4826·MAD) shows which signal can be
told from noise: a 3σ threshold means one noise pixel in about 700 crosses it (for a normal distribution).
The **variogram** γ(h) = ½·E[(U(x)−U(x+h))²] shows at what distance the noise stops being alike, i.e. the size of
atmospheric patches. Bowls (1–2 km) should be smaller than the filter σ, and the residual patches not larger than it.

In [ ]:
far = ~ndimage.binary_dilation(np.isfinite(U) & (ndimage.median_filter(np.nan_to_num(U), 3) < P0["VERT_ZONE_CM"]),
                               iterations=int(1000 / PX))
stable = np.isfinite(U) & far & ~water
u = U[stable]
sigma = 1.4826 * np.median(np.abs(u - np.median(u)))
print(f"Stable pixels: {u.size}; median {np.median(u):+.2f} cm/yr; σ (MAD) = {sigma:.1f} cm/yr")
print(f"→ 2σ = {-2*sigma:.1f}, 3σ = {-3*sigma:.1f}, 4σ = {-4*sigma:.1f} cm/yr (current threshold {P0['VERT_ZONE_CM']})")
# variogram from random pairs of pixels
yy, xx = np.nonzero(stable); n = min(len(yy), 6000)
sel = RNG.choice(len(yy), n, replace=False); yy, xx = yy[sel], xx[sel]
i, j = RNG.integers(0, n, 400_000), RNG.integers(0, n, 400_000)
h = np.hypot(yy[i] - yy[j], xx[i] - xx[j]) * PX / 1000
gam = 0.5 * (U[yy[i], xx[i]] - U[yy[j], xx[j]]) ** 2
hb = np.arange(0, 30.5, 1.0); idx = np.digitize(h, hb)
vg = pd.Series([np.median(gam[idx == k]) for k in range(1, len(hb))], index=hb[:-1] + 0.5)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(u, bins=np.arange(-40, 40, 1), color="#888"); [ax[0].axvline(-k*sigma, ls="--", c=c, label=f"−{k}σ")
    for k, c in [(2, "orange"), (3, "red"), (4, "darkred")]]
ax[0].axvline(P0["VERT_ZONE_CM"], c="k", lw=2, label="threshold −15"); ax[0].set_yscale("log"); ax[0].legend()
ax[0].set_xlabel("vertical in stable places, cm/yr"); ax[0].set_title("Vertical noise")
ax[1].plot(vg.index, vg.values, "o-"); ax[1].axvline(5, c="r", ls=":", label="HP_SIGMA_KM = 5")
ax[1].axvspan(1, 2, color="lime", alpha=.2, label="typical bowl size")
ax[1].set_xlabel("distance, km"); ax[1].set_ylabel("γ(h) (median), (cm/yr)²"); ax[1].legend(); ax[1].set_title("Noise variogram")
plt.tight_layout(); plt.savefig(OUT / "C_noise_variogram.png", dpi=130); plt.show()

## D. Null test: sign flips of the pairs
**Idea.** For each pixel subtract its mean rate, multiply the deviations of the single pairs by random signs ±1 and average.
Constant motion (a bowl) vanishes **exactly**, while noise and artefacts keep their size. Whatever the detector finds on such
"null" maps is a **false bowl**. Comparing the bowl counts on the real and the null maps gives an estimate of the **false
discovery rate (FDR)** for each threshold.

*Why this form:* the first version put the signs directly on the pair rates. With 8–9 pairs the signs almost never split
in half, so on average ~27 % of the motion stayed and the fast bowls (−70…−300 cm/yr) leaked into the null maps; the test
then showed a high FDR even at −30. The current version is free of that; the cell below checks it.

**Why the test is conservative (it overstates the FDR).** Neighbouring pairs share an acquisition, so in the real mean the
atmosphere of the shared acquisition partly cancels, and with random signs it does not. The null maps are therefore a little
noisier than the real one, and the true FDR is not larger than the estimate.

In [ ]:
THRS = [-8, -10, -12, -15, -20, -25, -30]
CONF = [1, 2, 3]
def null_maps():
    sA = RNG.choice([-1, 1], len(RA)); sD = RNG.choice([-1, 1], len(RD))
    vA0 = velocity_null(RA, sA); vD0 = on_grid(velocity_null(RD, sD), dD)
    vO0 = {k: on_grid(velocity_null(R_other[k], RNG.choice([-1, 1], len(R_other[k]))), stacks[k]) for k in others}
    return vertical(vA0, vD0), los_maps(vA0, vD0, vO0)

# control: no signal must remain at the centres of the real bowls in a null map
U0_chk, _ = null_maps()
cen = [(int(r.cy), int(r.cx)) for _, r in REF.head(10).iterrows()]
near = np.array([np.nanmean(U0_chk[max(y-2,0):y+3, max(x-2,0):x+3]) for y, x in cen]) if cen else np.array([])
print(f"Null control: null vertical at the top bowl centres = {np.round(near, 1)} cm/yr "
      f"(real: {np.round(REF.up_mean.head(10).values, 1)}); σ of the null in stable places = "
      f"{1.4826*np.nanmedian(np.abs(U0_chk[stable]-np.nanmedian(U0_chk[stable]))):.1f} cm/yr")

t = time.time(); res = []
for c in CONF:
    for th in THRS:
        p = {**P0, "VERT_ZONE_CM": th, "MIN_CONFIRM_TRACKS": c}
        res.append(dict(conf=c, thr=th, real=len(detect(U, L_real, p)), null=[]))
for k in range(N_PERM):
    U0, L0 = null_maps()
    for r in res:
        p = {**P0, "VERT_ZONE_CM": r["thr"], "MIN_CONFIRM_TRACKS": r["conf"]}
        r["null"].append(len(detect(U0, L0, p)))
    if (k + 1) % 10 == 0: print(f"  permutations {k+1}/{N_PERM} · {time.time()-t:.0f} s")
D = pd.DataFrame([dict(conf=r["conf"], thr=r["thr"], real=r["real"], null_mean=np.mean(r["null"]),
                       null_p95=np.percentile(r["null"], 95)) for r in res])
D["FDR"] = (D.null_mean / D.real.replace(0, np.nan)).clip(upper=1)
D.to_csv(OUT / "D_null_test.csv", index=False)
display(D.pivot(index="thr", columns="conf", values=["real", "null_mean", "FDR"]).round(2))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for c in CONF:
    d = D[D.conf == c]
    ax[0].plot(d.thr, d.real, "o-", label=f"real, ≥{c} track(s)")
    ax[0].plot(d.thr, d.null_mean, "x--", label=f"null, ≥{c} track(s)")
    ax[1].plot(d.thr, d.FDR, "o-", label=f"≥{c} track(s)")
for a in ax: a.axvline(P0["VERT_ZONE_CM"], c="k", ls=":"); a.set_xlabel("vertical threshold, cm/yr"); a.legend(fontsize=8)
ax[0].set_ylabel("number of bowls"); ax[1].set_ylabel("FDR estimate"); ax[1].axhline(0.1, c="r", ls="--")
ax[0].set_title("Real data vs null"); ax[1].set_title("False discovery rate (conservative)")
plt.tight_layout(); plt.savefig(OUT / "D_null_test.png", dpi=130); plt.show()
ok = D[(D.conf == P0["MIN_CONFIRM_TRACKS"]) & (D.FDR <= 0.10)]
print("Thresholds with FDR ≤ 10 % under the current confirmation rule:", sorted(ok.thr) or "none")
print("Note: with a small N_PERM the FDR estimates are noisy; use N_PERM ≥ 30 for a report.")

## E. Injected bowls (recall)
Take a **null** map (no motion) and inject Gaussian bowls of known strength (−10…−60 cm/yr at the centre) and size
(σ = 250–600 m) at random land places away from the real bowls, the same in all tracks, with each track's geometry. Count
which share the detector finds (centre closer than 500 m). This shows whether the threshold cuts off real bowls of the
strength we care about.

In [ ]:
AMPS = [-10, -20, -30, -45, -60]; SIGS = [250, 400, 600]; N_INJ_MAPS = 6; PER_MAP = 12
Yg, Xg = np.mgrid[0:U.shape[0], 0:U.shape[1]]
land = np.isfinite(U) & ~water & far
cand = np.argwhere(ndimage.binary_erosion(land, iterations=int(1000 / PX)))
uzA = dA.los_z.values; uzD = on_grid(dD.los_z.values, dD)
uzO = {k: on_grid(stacks[k].los_z.values, stacks[k]) for k in others}
rows = []
for th in [-10, -15, -20]:
    for m in range(N_INJ_MAPS):
        U0, L0 = null_maps()
        vA0, vD0 = L0[KA].copy(), L0[KD].copy(); vO0 = {k: L0[k].copy() for k in others}
        pts = cand[RNG.choice(len(cand), PER_MAP, replace=False)]
        truth = []
        for (cy, cx) in pts:
            a = RNG.choice(AMPS); s = RNG.choice(SIGS) / PX
            bowl = a * np.exp(-((Yg - cy) ** 2 + (Xg - cx) ** 2) / (2 * s ** 2))
            vA0 += bowl * uzA; vD0 += bowl * uzD                         # vertical motion only
            for k in others: vO0[k] += bowl * uzO[k]
            truth.append((cy, cx, a, s * PX))
        Ui = vertical(vA0, vD0)
        det_ = detect(Ui, los_maps(vA0, vD0, vO0), {**P0, "VERT_ZONE_CM": th})
        for (cy, cx, a, s) in truth:
            dmin = np.min(np.hypot(det_.cy - cy, det_.cx - cx)) * PX if len(det_) else np.inf
            rows.append(dict(thr=th, amp=a, sigma_m=s, found=dmin < 500))
E = pd.DataFrame(rows); E.to_csv(OUT / "E_injection.csv", index=False)
tab = E.groupby(["thr", "amp"]).found.mean().unstack(0)
display((tab * 100).round(0).astype(int).astype(str) + "%")
print("Injected bowls per (threshold, strength):", E.groupby(["thr", "amp"]).size().unstack(0).to_dict())
fig, ax = plt.subplots(figsize=(7, 4))
for th in tab.columns: ax.plot(tab.index, tab[th] * 100, "o-", label=f"threshold {th}")
ax.set_xlabel("bowl strength at the centre, cm/yr"); ax.set_ylabel("found, %"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Recall (injected bowls)"); plt.tight_layout(); plt.savefig(OUT / "E_injection.png", dpi=130); plt.show()

## F. Long-wavelength noise filter: does it eat the bowls?
Take one real pair, inject a bowl (−3 cm in 12 days, σ = 600 m) and apply the same masked Gaussian filter with several σ,
with the mask of "moving" pixels and without it. Look at the share of the bowl's amplitude that stays.

In [ ]:
def hp(Dp, sigma_km, mask_moving):
    sig = sigma_km * 1000 / PX
    w = np.isfinite(Dp) & ~mask_moving
    num = ndimage.gaussian_filter(np.where(w, Dp, 0), sig, mode="nearest")
    den = ndimage.gaussian_filter(w.astype("f8"), sig, mode="nearest")
    return Dp - np.where(den > 0.05, num / np.maximum(den, 1e-6), 0)
Dp = dA.disp.values[0].astype("f8") * 100                  # cm
cy, cx = cand[RNG.integers(len(cand))]
bowl = -3.0 * np.exp(-((Yg - cy) ** 2 + (Xg - cx) ** 2) / (2 * (600 / PX) ** 2))
Din = Dp + bowl
moving = ndimage.binary_dilation(np.abs(bowl) * 365.25 / 12 > 8, iterations=int(500 / PX))   # "motion" > 8 cm/yr (HP_MASK_CM) + 0.5 km
out = []
for s in [1, 2, 3, 5, 8, 12]:
    for use_mask in [True, False]:
        f = hp(Din, s, moving if use_mask else np.zeros_like(moving))
        f0 = hp(Dp, s, moving if use_mask else np.zeros_like(moving))
        rec = (f - f0)[cy, cx] / bowl[cy, cx]
        out.append(dict(sigma_km=s, mask=use_mask, amplitude_kept=round(float(rec), 2)))
F = pd.DataFrame(out).pivot(index="sigma_km", columns="mask", values="amplitude_kept")
F.columns = ["no mask", "with mask"]; display(F)
F.to_csv(OUT / "F_filter.csv")
print("Expected: with the mask the amplitude stays (~1.0) for every σ; without it, it drops at small σ.")

## G. `SPLIT_DEPTH_CM`: plateau of the bowl count

In [ ]:
G = []
for d in [2, 4, 6, 8, 10, 12, 15, 20, 25, 30]:
    r = detect(U, L_real, {**P0, "SPLIT_DEPTH_CM": d})
    G.append(dict(depth=d, n=len(r), n3=(r.n_conf >= 3).sum() if len(r) else 0))
G = pd.DataFrame(G); display(G.T)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(G.depth, G.n, "o-", label="all"); ax.plot(G.depth, G.n3, "s-", label="3 tracks")
ax.axvline(P0["SPLIT_DEPTH_CM"], c="k", ls=":"); ax.axvline(3 * sigma / 2, c="r", ls="--", label="~1.5σ of the smoothed noise (guide)")
ax.set_xlabel("min. saddle depth, cm/yr"); ax.set_ylabel("bowls"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "G_split.png", dpi=130); plt.show()
print("The threshold is justified if it sits on a plateau: small values cut bowls into noise, large ones split nothing.")

## H. Robustness of the conclusions
Change **one** parameter at a time around its default and look at (1) how many bowls are found, (2) how many of the
**reference 3-track bowls** (3 tracks, with the defaults) are found again (centre closer than 500 m), (3) the mean vertical
of the strongest bowl. The reference set here is the detector's own 3-track set, not the hand-picked "reliable" list of O2.

In [ ]:
ref3 = REF[REF.n_conf >= 3]
GRID = dict(VERT_ZONE_CM=[-10, -12, -15, -20, -25], CLUSTER_GAP_M=[200, 400, 600, 800], MAX_HOLE_KM2=[0.5, 1, 3, 5, 10],
            MAX_WATER_FRAC=[0.1, 0.2, 0.3, 0.5, 1.0], SPLIT_DEPTH_CM=[5, 10, 15, 20], MIN_ZONE_PX=[10, 20, 40],
            LOS_CONFIRM_CM=[-5, -8, -12], MIN_CONFIRM_TRACKS=[1, 2, 3])
H = []
for par, vals in GRID.items():
    for v in vals:
        r = detect(U, L_real, {**P0, par: v})
        if len(r) and len(ref3):
            d = np.hypot(r.cy.values[None] - ref3.cy.values[:, None], r.cx.values[None] - ref3.cx.values[:, None]) * PX
            rec = int((d.min(1) < 500).sum())
        else: rec = 0
        H.append(dict(parameter=par, value=v, default=(v == P0[par]), n_bowls=len(r),
                      n_3tracks=int((r.n_conf >= 3).sum()) if len(r) else 0,
                      ref_recovered=f"{rec}/{len(ref3)}", strongest_up=round(float(r.up_mean.min()), 1) if len(r) else None))
H = pd.DataFrame(H); H.to_csv(OUT / "H_robustness.csv", index=False)
pd.set_option("display.max_rows", 100); display(H)

## Summary: chosen value, justification and when it was chosen

In [ ]:
# the softest threshold for which it and all stricter ones have FDR ≤ 10 % (not a lucky single value)
best_thr = None
for _, r in D[D.conf == P0["MIN_CONFIRM_TRACKS"]].sort_values("thr").iterrows():
    if r.FDR <= 0.10 or np.isnan(r.FDR): best_thr = r.thr
    else: break
best_txt = f"FDR ≤ 10 % for all thresholds ≤ {best_thr:g}" if best_thr is not None else "NO threshold gave FDR ≤ 10 %: see table D"
# "when chosen" is as recorded by the analyst; SIGN follows notes/experiments.md O2
summary = pd.DataFrame([
  ["COH_MIN", 0.3, "before the results", "InSAR standard; test B: noise grows as γ → 0.3"],
  ["MIN_VALID_FRAC", 0.6, "before the results", "a pixel must be valid in most pairs"],
  ["SIGN", -1, "post-hoc (set so that bowls over mines sink; see experiments.md O2)", "same convention as the engine, which matched GNSS at other sites; not a test here"],
  ["MIN_COVERAGE", 0.30, "AFTER (0.15 → 0.30 because of D080)", f"test A: gap {lo:.2f}–{hi:.2f}, any value in it selects the same frames"],
  ["EDGE_ERODE_PX", 4, "AFTER (stripe at the D080 edge)", "known GUNW edge effect; does not touch bowls (≥ 5 px from the edge)"],
  ["DERAMP + HP_SIGMA_KM", 5, "AFTER (wide patches seen)", "test C (variogram) and F (with the mask, bowls do not shrink)"],
  ["VERT_ZONE_CM", -15, "AFTER (−10 → −15: 39 noisy zones)", f"test C: 3σ = {-3*sigma:.1f} (−15 ≈ {15/sigma:.1f}σ); test D: {best_txt}; test E: recall"],
  ["MIN_CONFIRM_TRACKS / LOS_CONFIRM_CM", "2 / −8", "before the results", "test D (FDR for 1/2/3 tracks), test H"],
  ["CLUSTER_GAP_M / MAX_HOLE_KM2", "400 / 3", "before the results (checked on synthetic data)", "test H: robustness"],
  ["MAX_WATER_FRAC + water mask", 0.30, "AFTER (Košťálovický pond)", "physics: water is incoherent; test H: robust over 0.1–1.0"],
  ["SPLIT_DEPTH_CM", 10, "before the results; agreed with the manual check of M02", "test G: plateau"],
  ["CONTAMINATION (Isolation Forest)", 0.03, "arbitrary", "does NOT affect the final bowls (they come from the vertical); IF only for LOS zones"],
], columns=["parameter", "value", "when chosen", "justification / test"])
summary.to_csv(OUT / "summary_thresholds.csv", index=False)
pd.set_option("display.max_colwidth", 120); display(summary)
print("Test files:", sorted(p.name for p in OUT.iterdir()))